# DealRoom S2P V3 — GRPO Training (V3 Enhanced)
## All improvements from analysis:
1. **Zero-centered sigmoid reward** (REWARD_GAIN=3.0, REWARD_SCALE=1.5) — eliminates 0.5 baseline bias
2. **Immediate milestone bonuses** — stage advance (+0.5), blocker resolve (+0.15), DPA to Legal (+0.3)
3. **Non-progress penalty** (-0.1) — penalizes inaction
4. **Diversity reward** (+0.05) — encourages varied actions
5. **POMDP noise** (σ=0.1, drop 30% weak signals) — true partial observability
6. **Pipe-delimited action format** — 100% parse success
7. **Multi-step generation** (depth=2) — model learns strategy, not just first move
8. **num_generations=16, discount=0.3** — stable gradients
9. **Hostile scenario veto grace rounds** (1 grace round)

## Algorithm choice: GRPO over PPO
| | GRPO | PPO |
|---|---|---|
| Value network | Not needed | Needs SimpleValueCritic (3-layer MLP) |
| T4 memory | ~2.3 GB model | +0.5 GB critic overhead |
| Stability | Stable from step 1 | Critic needs 100+ episodes to warm up |
| TRL integration | Native — GRPOTrainer expects reward_fn(prompts, completions) | Custom loop required |
| Verdict | **Use this** | Skip for this environment |

In [ ]:
# Cell 1: Install + clone

import subprocess, sys, os, warnings, json, re, random, threading, time
import numpy as np
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['DEALROOM_ENABLE_LLM_SUMMARY'] = '0'

# ── Install packages ────────────────────────────────────────────────────
subprocess.run(['pip', 'install', '-q', 'unsloth'], check=False)
subprocess.run(['pip', 'install', '-q', 'trl>=0.12.0', 'bitsandbytes', 'peft',
                'accelerate', 'datasets', 'numpy', 'matplotlib'], check=False)
n = subprocess.run(['pip', 'install', '-q', 'openenv-core'],
                   capture_output=True, text=True)
if n.returncode != 0:
    print('openenv-core not on PyPI — injecting stub')

# ── openenv stub (if real package unavailable) ──────────────────────────────────
import types as _types
if 'openenv' not in sys.modules:
    _ov = _types.ModuleType('openenv')
    _oc = _types.ModuleType('openenv.core')
    _oct = _types.ModuleType('openenv.core.client_types')
    _oce = _types.ModuleType('openenv.core.env_server')
    _ocet = _types.ModuleType('openenv.core.env_server.types')
    class _EnvClient: pass
    class _StepResult: pass
    _EnvClient.__module__ = 'openenv.core'
    _StepResult.__module__ = 'openenv.core.env_server.types'
    _ocet.StepResult = _StepResult
    _oce.types = _ocet
    _oc.env_server = _oce
    _oc.client_types = _oct
    _ov.core = _oc
    sys.modules['openenv'] = _ov
    sys.modules['openenv.core'] = _oc
    sys.modules['openenv.core.client_types'] = _oct
    sys.modules['openenv.core.env_server'] = _oce
    sys.modules['openenv.core.env_server.types'] = _ocet

# ── Clone / pull deal_room_S2P ─────────────────────────────────
REPO_DIR = '/content/deal_room_S2P'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/akshaypulla/deal_room_S2P.git',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
sys.path.insert(0, REPO_DIR)

# ── Import deal_room_S2P ─────────────────────────────────
import deal_room_S2P
from deal_room_S2P.environment.dealroom_v3 import DealRoomV3S2P
from deal_room_S2P.environment.prompts import (
    build_situation_prompt,
    parse_action_text,
    build_stakeholder_prompt,
)
from deal_room_S2P.environment.constants import REWARD_WEIGHTS

print('All imports OK')

In [ ]:
# Cell 2: Environment smoke test with new features

print('=== SMOKE TEST ===')

env = DealRoomV3S2P()
obs = env.reset(seed=42, task_id='aligned')
print(f'reset() OK. Stage: {obs.deal_stage}, Round: {obs.round_number}')
print(f'Stakeholders: {list(obs.stakeholders.keys())}')

# Test A: send_document to Legal with DPA (high-value action — addresses CVaR veto risk)
a1_text = 'send_document Legal dpa | Our Data Protection Agreement covers GDPR Article 28 processor obligations.'
a1 = parse_action_text(a1_text)
obs2, r1, done1, info1 = env.step(a1)
print(f'\nAction A (DPA to Legal): {a1_text[:60]}')
print(f'  reward={r1:.4f}  done={done1}  stage={obs2.deal_stage}')
print(f'  reward_components={info1.get("reward_components", {})}')  

# Test B: bare message (low-value action — should get lower reward)
env.reset(seed=42, task_id='aligned')
a2_text = 'direct_message Legal | We take compliance seriously.'
a2 = parse_action_text(a2_text)
obs3, r2, done2, info2 = env.step(a2)
print(f'\nAction B (bare message): {a2_text[:60]}')
print(f'  reward={r2:.4f}  done={done2}  stage={obs3.deal_stage}')
print(f'  reward_components={info2.get("reward_components", {})}')  

# Test C: pipe-delimited format round-trips
texts = [
    'send_document Legal dpa | Please review our DPA.',
    'direct_message Finance | We offer competitive pricing.',
    'concession Finance | price=175000',
    'group_proposal | All parties are aligned on key terms.',
    'exec_escalation | We need executive review.',
    # Also test old format (backward compat)
    'send_document Legal DPA Please review our DPA. ###',
    'direct_message Finance We offer competitive pricing. ###',
]
print('\nParse round-trips:')
for t in texts:
    parsed = parse_action_text(t)
    print(f'  {t[:55]!r} \u2192 {parsed.action_type} target={parsed.target_ids}')

# Test D: hostile scenario has grace rounds
env.reset(seed=42, task_id='hostile_acquisition')
print(f'\nHostile scenario veto_grace_rounds: {env._scenario.veto_grace_rounds}')

env.close()
print('\nSMOKE TEST PASSED')

In [ ]:
# Cell 3: Reward function — minimal correct GRPO reward
_reward_log = []

from deal_room_S2P.environment.minimal_grpo_reward import MinimalDealRoomReward, reward_fn
print(f'Reward function: {reward_fn.__name__}')

# Sanity check
valid_outcome = reward_fn(["prompt"], ["send_document Legal dpa | Our DPA covers GDPR obligations."])
invalid_outcome = reward_fn(["prompt"], ["xyzabc123 invalid format no keywords here at all"])
print(f'Sanity — valid: {valid_outcome[0]:.4f}  invalid: {invalid_outcome[0]:.4f}  (should be -0.5)')
if invalid_outcome[0] != -0.5:
    raise RuntimeError(f'Parse penalty broken! Got {invalid_outcome[0]}, expected -0.5')
print('Reward function ready.')


In [ ]:
# Cell 4: Model loading — Unsloth QLoRA on T4 (15GB VRAM)

import torch
from unsloth import FastLanguageModel
from transformers import GenerationConfig

MAX_SEQ_LENGTH = 2048
MODEL_NAME     = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
model.config.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing=True, random_state=42,
)

model.generation_config = GenerationConfig(
    max_new_tokens=40,
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    eos_token_id=[tokenizer.eos_token_id],
    do_sample=True,
    temperature=0.8,
    use_cache=True,
)
if hasattr(model.generation_config, 'max_length'):
    model.generation_config.max_length = None

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
mem_gb    = torch.cuda.memory_allocated() / 1e9
print(f'Model: {MODEL_NAME}')
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print(f'GPU allocated: {mem_gb:.2f} GB')
print('Memory OK' if mem_gb < 10 else 'WARNING: > 10GB — may OOM. Try r=8.')

In [ ]:
# Cell 5: System prompt + dataset (strict format with negative examples)

SYSTEM_PROMPT = '''You are an enterprise B2B sales executive negotiating a deal.

OUTPUT EXACTLY ONE LINE in this format:
  send_document <target> <doc_type> | <message>
  direct_message <target> | <message>
  concession <target> | <term>=<value>
  group_proposal | <message>
  exec_escalation | <message>

✅ CORRECT outputs:
send_document Legal dpa | Our DPA covers GDPR Article 28 processor obligations.
send_document Finance roi_model | 3-year NPV shows $2.1M return, 14-month payback.
concession Finance | price=175000
direct_message TechLead | Our REST API has full OpenAPI documentation and sandbox access.
group_proposal | All stakeholders are aligned on key terms.
exec_escalation | Ready for executive sign-off.

❌ WRONG outputs (will be heavily penalized):
For example: send_document Legal dpa | ...
To address the issue, we should send a document to Legal.
Here is my action: send_document Legal dpa | ...
I will send the DPA to Legal.
Let me send a document to Finance about the ROI model.
Sure, I can help with that by sending a document.

Rules:
- Use LOWERCASE: send_document, NOT Send Document or SEND DOCUMENT
- Use | (pipe) as delimiter — NEVER use ###
- Do NOT write introductions like "For example:" or "To address..."
- Do NOT explain your choice
- Do NOT output JSON, markdown, or multi-line text
- Output ONLY the action — one line, nothing else

Valid targets: Legal, Finance, TechLead, Procurement, Operations, ExecSponsor
Valid doc types: dpa, security_cert, roi_model, implementation_timeline, compliance_report
'''

def build_prompt(obs):
    try:
        situation = build_situation_prompt(obs)
    except Exception:
        situation = f"Round {obs.round_number}/{obs.max_rounds}, Stage: {obs.deal_stage}\n"
        engagement = obs.engagement_level or {}
        for sid, eng in engagement.items():
            situation += f"{sid}: engagement={int(eng*100)}%\n"
    return f"{SYSTEM_PROMPT}\n\n{situation}"

def build_dataset(n_samples=80, seeds=None, tasks=None):
    if seeds is None: seeds = list(range(42, 42 + n_samples))
    if tasks is None: tasks = ['aligned'] * 20 + ['conflicted'] * 30 + ['hostile_acquisition'] * 30
    pairs = []
    for i in range(n_samples):
        env = DealRoomV3S2P()
        task_id = tasks[i % len(tasks)]
        obs = env.reset(seed=seeds[i], task_id=task_id)
        prompt = build_prompt(obs)
        if task_id == 'hostile_acquisition':
            pos = 'send_document Legal dpa | Our DPA covers GDPR obligations and provides full liability protection.'
        elif task_id == 'conflicted':
            pos = 'send_document Finance roi_model | 3-year analysis shows strong ROI with protected downside.'
        else:
            pos = 'send_document TechLead security_cert | ISO 27001 certification covers all customer systems.'
        baseline = 'direct_message Legal | We are committed to this deal.'
        pairs.append({'prompt': prompt, 'completion': pos})
        pairs.append({'prompt': prompt, 'completion': baseline})
        env.close()
    from datasets import Dataset
    return Dataset.from_list(pairs)

train_dataset = build_dataset(n_samples=80)
print(f'Dataset: {len(train_dataset)} prompt-completion pairs')
print(f'Sample lengths: {[len(tokenizer(s["prompt"]+" "+s["completion"])) for s in train_dataset.select(range(3))]}')


In [ ]:
# Cell 6: GRPOConfig + GRPOTrainer

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='./dealroom_s2p_grpo_v3',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=60,
    seed=42,
    max_prompt_length=1600,
    max_completion_length=40,
    num_generations=16,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    epsilon=0.2,
    beta=0.01,
    learning_rate=5e-6,
    gradient_checkpointing=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=3,
    save_steps=30,
    save_total_limit=2,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
print('GRPOTrainer initialized')
print(f'  max_steps={training_args.max_steps}')
print(f'  num_generations={training_args.num_generations}')
print(f'  temperature={training_args.temperature}, top_p={training_args.top_p}')
print(f'  beta={training_args.beta}')


In [ ]:
# Cell 7: Live debug callback

from transformers import TrainerCallback

class DealRoomDebugCallback(TrainerCallback):
    def __init__(self):
        self._start_reward = None
        self._low_std_count = 0
        self._start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or 'reward' not in logs:
            return
        step    = state.global_step
        reward  = logs.get('reward', 0)
        rstd    = logs.get('reward_std', 0)
        kl      = logs.get('kl', 0)
        clip    = logs.get('clipped_ratio', 0)
        entropy = logs.get('entropy', 0)
        elapsed = time.time() - self._start_time

        if self._start_reward is None:
            self._start_reward = reward

        std_status  = 'HEALTHY' if rstd >= 0.05 else ('LOW' if rstd >= 0.02 else 'COLLAPSED')
        kl_status   = 'LEARNING' if kl > 1e-5 else 'FLAT'
        clip_status = 'OK' if clip < 0.2 else 'HIGH'
        ent_status  = 'DIVERSE' if entropy > 0.5 else ('NARROW' if entropy > 0.1 else 'COLLAPSED')

        print(f'\nStep {step}/60:')
        print(f'  reward:      {reward:.4f} (Δ from start: {reward-self._start_reward:+.4f})')
        print(f'  reward_std:  {rstd:.4f}  [{std_status}]')
        print(f'  entropy:     {entropy:.4f}  [{ent_status}]')
        print(f'  kl:          {kl:.6f}  [{kl_status}]')
        print(f'  clipped:     {clip:.4f}  [{clip_status}]')
        print(f'  elapsed:     {elapsed:.0f}s')

        # Action diversity from reward log
        if _reward_log:
            recent = _reward_log[-16:]
            reward_vals = [r['r'] for r in recent]
            r_std = float(np.std(reward_vals)) if len(reward_vals) > 1 else 0
            print(f'  recent_r:    min={min(reward_vals):.3f}  max={max(reward_vals):.3f}  std={r_std:.3f}')

        # Parse fail logging from reward_fn
        dbg = reward_fn.debug_summary() if hasattr(reward_fn, 'debug_summary') else {}
        pf = dbg.get('parse_fails', len(_parse_fails) if '_parse_fails' in dir() else 0)
        if pf:
            print(f'  parse_fails: {pf} total')
            lft = dbg.get('last_fail_text', 'unknown')
            if lft:
                print(f'  last_fail:   {lft[:80]}')

        # Mode collapse detector
        if rstd < 0.02:
            self._low_std_count += 1
            if self._low_std_count >= 3:
                print('  WARNING: reward_std < 0.02 for 3 consecutive — possible mode collapse!')
        else:
            self._low_std_count = 0


In [ ]:
# Cell 8: Train
callback = DealRoomDebugCallback()
trainer.add_callback(callback)
trainer.train()
print('Training complete!')


In [ ]:
# Cell 9: Training curves (4 subplots)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

log = trainer.state.log_history
steps, rewards, rstds, kls, clips, entropies = [], [], [], [], [], []
for e in log:
    if 'reward' in e and 'step' in e:
        steps.append(e['step'])
        rewards.append(e['reward'])
        rstds.append(e.get('reward_std', 0))
        kls.append(e.get('kl', 0))
        clips.append(e.get('clipped_ratio', 0))
        entropies.append(e.get('entropy', 0))
if not steps:
    print('No logged steps — training may not have run yet')
else:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle('DealRoom S2P V3 Enhanced — GRPO Training', fontsize=14, fontweight='bold')
    
    ax = axes[0, 0]
    ax.plot(steps, rewards, 'o-', color='steelblue', lw=2)
    lo = [r-s for r,s in zip(rewards, rstds)]
    hi = [r+s for r,s in zip(rewards, rstds)]
    ax.fill_between(steps, lo, hi, alpha=0.2, color='steelblue')
    d = rewards[-1] - rewards[0]
    ax.text(0.05, 0.05, f'Δ={d:+.3f}', transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))
    ax.set_title('Reward (mean ± std)'); ax.set_xlabel('Step'); ax.grid(alpha=0.3)
    
    ax = axes[0, 1]
    ax.plot(steps, entropies, 's-', color='orange', lw=2)
    ax.set_title('Policy Entropy (rising = diverse)'); ax.set_xlabel('Step')
    ax.grid(alpha=0.3)
    
    ax = axes[1, 0]
    ax.plot(steps, kls, '^-', color='mediumpurple', lw=2)
    ax.set_title('KL divergence (rising = learning)'); ax.set_xlabel('Step'); ax.grid(alpha=0.3)
    
    ax = axes[1, 1]
    ax.plot(steps, clips, 'D-', color='coral', lw=2)
    ax.axhline(0.2, color='red', ls='--', alpha=0.5, label='clip limit (0.2)')
    ax.set_title('Clipping ratio'); ax.set_xlabel('Step')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    
    plt.tight_layout()
    Path('./dealroom_s2p_grpo_v3').mkdir(exist_ok=True)
    plt.savefig('./dealroom_s2p_grpo_v3/training_curves.png', dpi=150)
    print('Curves saved: ./dealroom_s2p_grpo_v3/training_curves.png')

In [ ]:
# Cell 10: Post-training verification

FastLanguageModel.for_inference(model)

def run_episode(task='aligned', seed=42, use_baseline=False, max_steps=10):
    env = DealRoomV3S2P()
    obs = env.reset(seed=seed, task_id=task)
    total, outcome = 0.0, 'timeout'
    
    for _ in range(max_steps):
        if use_baseline:
            act_text = 'direct_message Legal | We are committed to this deal.'
        else:
            prompt = build_prompt(obs)
            inps = tokenizer(prompt, return_tensors='pt',
                            truncation=True, max_length=1800).to(model.device)
            with torch.no_grad():
                out = model.generate(**inps, max_new_tokens=128, temperature=0.4,
                                      do_sample=True, pad_token_id=tokenizer.eos_token_id)
            act_text = tokenizer.decode(out[0][inps.input_ids.shape[1]:],
                                         skip_special_tokens=True).strip()
        
        try:
            act = parse_action_text(act_text)
        except Exception:
            act = parse_action_text('direct_message Legal | We remain committed.')
        
        obs, r, done, info = env.step(act)
        total += r
        if done:
            outcome = info.get('terminal_outcome', 'done') if isinstance(info, dict) else 'done'
            break
    env.close()
    return total, outcome

# Test 1: Trained vs baseline
print('=== Test 1: Trained vs Baseline ===')
scenarios = [('aligned',42), ('aligned',43), ('conflicted',44),
             ('conflicted',45), ('hostile_acquisition',46)]
trained_r, baseline_r = [], []
print(f'{"Scenario":<25s}  {"Trained":>10s}  {"Baseline":>10s}  {"Delta":>8s}')
print('-' * 60)
for task, seed in scenarios:
    tr, _ = run_episode(task, seed, use_baseline=False)
    br, _ = run_episode(task, seed, use_baseline=True)
    trained_r.append(tr)
    baseline_r.append(br)
    delta = tr - br
    print(f'{task:<25s}  {tr:>10.4f}  {br:>10.4f}  {delta:>+8.4f}')

avg_delta = np.mean([t-b for t,b in zip(trained_r, baseline_r)])
print(f'\nAverage trained vs baseline delta: {avg_delta:+.4f}')

# Test 2: Action diversity
print('\n=== Test 2: Action Diversity ===')
action_types = []
for task, seed in scenarios:
    for _ in range(3):
        _, outcome = run_episode(task, seed, use_baseline=False)
        for log_entry in _reward_log[-8:]:
            if log_entry['t'] == task:
                action_types.append(log_entry.get('action_type', 'unknown'))
                break
unique_actions = len(set(action_types)) if action_types else 0
print(f'Unique action types seen: {unique_actions} (higher = more diverse)')

# Test 3: Hostile scenario grace round
print('\n=== Test 3: Hostile Scenario Grace Round ===')
env = DealRoomV3S2P()
obs = env.reset(seed=42, task_id='hostile_acquisition')
print(f'Veto grace rounds: {env._scenario.veto_grace_rounds}')
print(f'First round veto check skipped: {env._round_number <= env._scenario.veto_grace_rounds}')
env.close()

In [ ]:
# Cell 11: Save model + metrics

from pathlib import Path
import json as _json

out_dir = './dealroom_s2p_grpo_v3/final'
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print(f'Model saved: {out_dir}')

# Extract final metrics
log_entries = [e for e in trainer.state.log_history if 'reward' in e]
if log_entries:
    metrics = {
        'algorithm': 'GRPO Enhanced',
        'model': 'Qwen2.5-3B-Instruct QLoRA r=16',
        'max_steps': 60,
        'num_generations': 16,
        'entropy_coef': 0.01,
        'discount': 0.3,
        'n_seed_samples': 3,
        'reward_start': log_entries[0]['reward'],
        'reward_end': log_entries[-1]['reward'],
        'reward_delta': log_entries[-1]['reward'] - log_entries[0]['reward'],
        'reward_std_final': log_entries[-1].get('reward_std', 0),
        'kl_final': log_entries[-1].get('kl', 0),
        'entropy_final': log_entries[-1].get('entropy', 0),
        'improvements': [
            'zero-centered sigmoid reward (REWARD_GAIN=3.0)',
            'milestone bonuses (stage advance +0.5, DPA +0.3)',
            'non-progress penalty (-0.1)',
            'diversity reward (+0.05)',
            'POMDP noise (true partial observability)',
            'pipe-delimited action format',
            'multi-step generation (depth=2)',
            'hostile scenario veto grace rounds',
        ],
    }
    Path('./dealroom_s2p_grpo_v3').mkdir(exist_ok=True)
    with open('./dealroom_s2p_grpo_v3/metrics.json', 'w') as f:
        _json.dump(metrics, f, indent=2)
    print('Metrics saved')
    print(f'reward_delta: {metrics["reward_delta"]:+.4f}')
    print(f'entropy_final: {metrics["entropy_final"]:.4f} (higher = more diverse)')

# Uncomment to push to HuggingFace Hub
# from huggingface_hub import login; login()
# model.push_to_hub('yourusername/dealroom-s2p-qwen-3b-grpo-v3')
# tokenizer.push_to_hub('yourusername/dealroom-s2p-qwen-3b-grpo-v3')